# Check concatenated results from Gelato

- author : Sylvie Dagoret-Campagne
- creation date : 2024-08-29
- update : 2024-05-29
- last update : 2024-08-29



In [ ]:
import pandas as pd
from astropy.io import fits
from astropy.table import Table
import re
import os
import gelato
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
import matplotlib as mpl
mpl.rcParams['font.size'] = 16

## Config

In [ ]:
version = "v3"
path_params = f"./ExampleParametersFitInNb_{version}.json"

# Create Parameters dictionary
params_gel = gelato.ConstructParams.construct(path_params)

In [ ]:
file_fit_results_fn = "GELATO-results.fits"

In [ ]:
OutFolder = params_gel["OutFolder"]

In [ ]:
file_fit_results_fullfn=os.path.join(OutFolder,file_fit_results_fn)

In [ ]:
flag_save_filelist = False

## Read full object list

In [ ]:
t = Table.read(file_fit_results_fullfn)

In [ ]:
df = t.to_pandas()

In [ ]:
for col in df.columns:
    if "Flux_err" in col:
        print(col)

In [ ]:
the_chi2_values = list(df.rChi2.values)

In [ ]:
title = "GETATO Fit (Fors2) Reduced Chi2"
xlabel = "$\chi^2/Ndf$"
fig, ax = plt.subplots(figsize=(6, 4), facecolor='white',
                       layout='constrained')
ax.hist(the_chi2_values ,bins=100,range=(0,5.));
ax.grid()
ax.set_title(title)
ax.set_xlabel(xlabel)

In [ ]:
groups = params_gel['EmissionGroups']

In [ ]:
for group in groups:
    g_name = group['Name']
    for species in group['Species']:
        s_name = species['Name']
        all_lines = species['Lines']
        for line in all_lines:
            wl = line['Wavelength']
            l_tag =  g_name +'_' + s_name +'_' + str(wl)
            print(l_tag)
            if g_name == "AGN" and s_name == "[OIII]":
                c_tag =  'Outflow' +'_' + s_name + '_Outflow_' + str(wl)
                print(c_tag)
            if g_name == "Balmer" and s_name == "HI":
                c_tag =  g_name +'_' + s_name + '_Broad_' + str(wl)
                print(c_tag)
            

In [ ]:
def PlotFlux(data,title,xlabel):
    fig, ax = plt.subplots(figsize=(5, 4), facecolor='white',layout='constrained')
    nbins = 100
    
    #ax.hist(data ,bins=nbins);
    
    hist, bins = np.histogram(data, bins=nbins)
    logbinmin = np.log10(bins[0])
    logbinmax = np.log10(bins[-1])
    if np.isnan(logbinmin)  or np.isnan(logbinmax):
        # plot on linear scale
        ax.hist(data, bins=nbins)
    else:
        logbins = np.logspace(np.log10(bins[0]),np.log10(bins[-1]),len(bins))
        ax.hist(data, bins=logbins)
        plt.gca()
        plt.xscale('log')
    
    
    ax.grid()
    ax.set_title(title)
    ax.set_xlabel(xlabel)

    plt.show()

## Selection

In [ ]:
cut = df.rChi2.values< 2.0

In [ ]:
for group in groups:
    g_name = group['Name']
    for species in group['Species']:
        s_name = species['Name']
        all_lines = species['Lines']
        for line in all_lines:
            wl = line['Wavelength']
            c_tag =  g_name +'_' + s_name +'_' + str(wl)
            c_name = c_tag +  "_Flux"
            c_name_err = c_tag +  "_Flux_err"
            values = df[c_name][cut].values
            values_err = df[c_name_err][cut].values
            s_n = values/values_err
            cut_2 = (np.abs(s_n)> 5.) & ( ~np.isnan(values) & (values !=0))
            title = c_tag
            xlabel = c_name
            val_to_plot = values[cut_2]
            if len(val_to_plot)>0:
                PlotFlux(val_to_plot,title,xlabel)
 
            if g_name == "AGN" and s_name == "[OIII]":
                c_tag =  'Outflow' +'_' + s_name + '_Outflow_' + str(wl)
                c_name = c_tag +  "_Flux"
                c_name_err = c_tag +  "_Flux_err"
                values = df[c_name][cut].values
                values_err = df[c_name_err][cut].values
                s_n = values/values_err
                cut_2 = (np.abs(s_n)> 5.) & (~np.isnan(values) & (values !=0.0))
                title = c_tag
                xlabel = c_name
                val_to_plot = values[cut_2]
                if len(val_to_plot)>0:
                    PlotFlux(val_to_plot,title,xlabel)
                
            if g_name == "Balmer" and s_name == "HI":
                c_tag =  g_name +'_' + s_name + '_Broad_' + str(wl)
                c_name = c_tag +  "_Flux"
                c_name_err = c_tag +  "_Flux_err"
                values = df[c_name][cut].values
                values_err = df[c_name_err][cut].values
                s_n = values/values_err
                cut_2 = (np.abs(s_n)> 5.) & (~np.isnan(values) & (values !=0.0 ))
                title = c_tag
                xlabel = c_name
                val_to_plot = values[cut_2]
                if len(val_to_plot)>0:
                    PlotFlux(val_to_plot,title,xlabel)

## BPT Plots

In [ ]:
df_sel = df[cut]

In [ ]:
oiii_1 = df_sel['AGN_[OIII]_5008.24_Flux']
oiii_2 = df_sel['AGN_[OIII]_4960.295_Flux']
#oiii_3 = t['AGN_[OIII]_4364.436_Flux']

nii_1 = df_sel['AGN_[NII]_6585.27_Flux']
nii_2 = df_sel['AGN_[NII]_6549.86_Flux']

df_sel["ha"] = df_sel['Balmer_HI_6564.61_Flux']
df_sel["hb"] = df_sel['Balmer_HI_4862.68_Flux']

In [ ]:
def sum_oiii(row):
    flux1 = row['AGN_[OIII]_5008.24_Flux']
    flux1_err = row['AGN_[OIII]_5008.24_Flux_err']
    sn1 = flux1/flux1_err
    
    flux2 = row['AGN_[OIII]_4960.295_Flux']
    flux2_err = row['AGN_[OIII]_4960.295_Flux_err']
    sn2 = flux2/flux2_err


    flux_sum = 0.0
    flux_err_2 = 0.0

    if (not np.isnan(sn1)) and (sn1>2.0):
        flux_sum += flux1
        flux_err_2 += flux1_err**2
        
    if (not np.isnan(sn2)) and (sn2>2.0):
        flux_sum += flux2
        flux_err_2 += flux2_err**2
        
    if flux_sum >0:
        return flux_sum,np.sqrt(flux_err_2)
    else:
        return np.nan,np.nan

In [ ]:
def sum_nii(row):
    flux1 = row['AGN_[NII]_6585.27_Flux']
    flux1_err = row['AGN_[NII]_6585.27_Flux_err']
    sn1 = flux1/flux1_err
    
    flux2 = row['AGN_[NII]_6549.86_Flux']
    flux2_err = row['AGN_[NII]_6549.86_Flux_err']
    sn2 = flux2/flux2_err


    flux_sum = 0.0
    flux_err_2 = 0.0

    if (not np.isnan(sn1)) and (sn1>2.0):
        flux_sum += flux1
        flux_err_2 += flux1_err**2
        
    if (not np.isnan(sn2)) and (sn2>2.0):
        flux_sum += flux2
        flux_err_2 += flux2_err**2
        
    if flux_sum >0:
        return flux_sum,np.sqrt(flux_err_2)
    else:
        return np.nan,np.nan

In [ ]:
df_sel[["oiii","oiii_err"]]=df_sel.apply(sum_oiii,axis=1,result_type="expand")
df_sel[["nii","nii_err"]]=df_sel.apply(sum_nii,axis=1,result_type="expand")

In [ ]:
# Check
#df_sel["oiii"].dropna()
#df_sel["nii"].dropna()
df_sel["nii_ha"] = df_sel["nii"]/df_sel["ha"]
df_sel["oiii_hb"] = df_sel["oiii"]/df_sel["hb"]

In [ ]:
# Create figure
fig, ax = plt.subplots(figsize=(6,6))

# Kewley+ Line
x = np.logspace(-1.5,0.05,100)
y = 10**(0.61/(np.log10(x) - 0.05) + 1.3)
ax.plot(x,y,color='gray',ls='--',label='Kauffman+03')

# Kauffman+ Line
x = np.logspace(-1.5,0.47,100)
y = 10**(0.61/(np.log10(x) - 0.47) + 1.18)
ax.plot(x,y,color='gray',ls='-',label='Kewley+01')

# Plot BPT
nii_ha = df_sel["nii_ha"].values
oiii_hb = df_sel["oiii_hb"].values
ax.scatter(nii_ha,oiii_hb,color='k',label='Bootstraps',edgecolors='none',alpha=1.0)
#x,y,xerr,yerr = np.median(nii/ha),np.median(oiii/hb),np.std(nii/ha),np.std(oiii/hb)
#ax.errorbar(x,y,xerr=xerr,yerr=yerr,color='r',label='Average')
#ax.legend()

# Axis limits
ax.set(xlim=[1e-1,1e1],ylim=[1e-1,2e1])

# Axis labels
ax.set(xlabel=r'[NII]/H$\alpha$',ylabel=r'[OIII]/H$\beta$')

# Axis scale
ax.set(yscale='log',xscale='log')

# Show figure
ax.set_title("BPT for Fors2")